# Module 02: Customers EDA & Cleaning

### 1. Load Raw Data (Bronze Layer)

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path("../data")
RAW_DATA_PATH = DATA_DIR / "olist_customers_dataset.csv"

# Ensure the file exists before loading
assert RAW_DATA_PATH.exists(), f"Source file not found at: {RAW_DATA_PATH.resolve()}"

customers = pd.read_csv(RAW_DATA_PATH)
print(f"Loaded raw dataset shape: {customers.shape}")
customers.head(3)

Loaded raw dataset shape: (99441, 5)


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP


### 2. Data Audit & Missing Values Check
Analyzing data types, missing value distribution, and column cardinality.

In [2]:
audit_df = pd.DataFrame(
    {
        "Data Type": customers.dtypes,
        "Missing Values": customers.isna().sum(),
        "Missing %": (customers.isna().mean() * 100).round(2),
        "Unique Values": customers.nunique(),
    }
)

display(audit_df)

,Data Type,Missing Values,Missing %,Unique Values
customer_id,object,0,0.0,99441
customer_unique_id,object,0,0.0,96096
customer_zip_code_prefix,int64,0,0.0,14994
customer_city,object,0,0.0,4119
customer_state,object,0,0.0,27


### 3. Text Standardization & Silver Layer Initialization

To eliminate discrepancies caused by formatting inconsistencies, we standardize all text fields:
- **String Cleaning:** Strip unwanted whitespaces and enforce proper capitalization for city names (`.title()`).
- **State Code Harmonization:** Ensure all state abbreviations conform strictly to uppercase ISO standards.
- **ZIP Code Formatting:** Convert postal code prefixes into 5-digit zero-padded strings to prepare for accurate spatial and geographic operations.

In [3]:
customers_clean = customers.copy()

placeholders = ["", "-", "n/a", "null", "undefined", "unknown"]

customers_clean["customer_city"] = customers_clean["customer_city"].replace(placeholders, np.nan)
customers_clean["customer_city"] = customers_clean["customer_city"].str.split().str.join(" ").str.title()
customers_clean["customer_state"] = customers_clean["customer_state"].astype(str).str.strip().str.upper()
customers_clean["customer_zip_code_prefix"] = customers_clean["customer_zip_code_prefix"].astype(str).str.zfill(5)

display(customers_clean.head(3))

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,Franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,Sao Bernardo Do Campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,Sao Paulo,SP


In [4]:
customers_clean.isna().sum()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

### 4. Data Type Optimization

To transition our dataset from the Bronze layer to the Silver layer, we enforce robust data types:
- **Identifiers & Text (`string`):** Standardize `customer_id`, `customer_unique_id`, and `customer_city` into explicit string types for schema consistency and relational integrity.
- **Postal Codes (`string`):** Enforce string format for `customer_zip_code_prefix` to securely preserve leading zeros.
- **Categories (`category`):** Efficient for low-cardinality text columns (like `customer_state`), reducing memory overhead and accelerating group-by operations.

In [5]:
customers_clean["customer_id"] = customers_clean["customer_id"].astype("string")
customers_clean["customer_unique_id"] = customers_clean["customer_unique_id"].astype("string")
customers_clean["customer_zip_code_prefix"] = customers_clean["customer_zip_code_prefix"].astype("string")
customers_clean["customer_city"] = customers_clean["customer_city"].astype("string")
customers_clean["customer_state"] = customers_clean["customer_state"].astype("category")

display(customers_clean.dtypes.to_frame(name="Silver Layer Data Type"))

,Silver Layer Data Type
customer_id,string[python]
customer_unique_id,string[python]
customer_zip_code_prefix,string[python]
customer_city,string[python]
customer_state,category


### 5. Data Integrity Checks & Anomaly Detection

Ensuring logical consistency, validating business rules, and inspecting value distributions to isolate structural anomalies and outliers:
- **ZIP Code Range Validation:** Checking for postal codes outside the valid 5-digit range.
- **City Name Sanitization:** Screening for empty, missing, or placeholder city values (e.g., `-`, `n/a`, `unknown`).
- **State Code Harmonization:** Verifying that all state abbreviations strictly match official Brazilian state codes.

In [6]:
valid_brazil_states = [
    'AC', 'AL', 'AP', 'AM', 'BA', 'CE', 'DF', 'ES', 'GO', 'MA', 
    'MT', 'MS', 'MG', 'PA', 'PB', 'PR', 'PE', 'PI', 'RJ', 'RN', 
    'RS', 'RO', 'RR', 'SC', 'SP', 'SE', 'TO'
]

audit_checks = {
    "Invalid ZIP code range": 
        (customers_clean["customer_zip_code_prefix"].astype(int) < 1000) | 
        (customers_clean["customer_zip_code_prefix"].astype(int) > 99999),

    "Invalid, empty or placeholder city name": 
        customers_clean["customer_city"].str.strip().str.lower().isin(["", "n/a", "null", "-", "nan", "undefined", "unknown", "none"]),

    "Unrecognized Brazilian state code": 
        ~customers_clean["customer_state"].isin(valid_brazil_states)
}

for label, condition in audit_checks.items():
    print(f"{label}: {condition.sum()}")


Invalid ZIP code range: 0
Invalid, empty or placeholder city name: 0
Unrecognized Brazilian state code: 0


### 6. Silver Layer Validation & Export
Comparing Bronze vs. Silver metrics (shape, types, and record count) and persisting the dataset to Parquet.

In [7]:
print("Pipeline Summary: Bronze vs Silver Layer")
print(f"Original shape (Bronze): {customers.shape}")
print(f"Cleaned shape (Silver): {customers_clean.shape}")
print(f"Total rows dropped: {len(customers) - len(customers_clean):,}")

# Side-by-side comparison of data types
dtypes_comparison = pd.DataFrame({
    'Bronze (Raw)': customers.dtypes,
    'Silver (Clean)': customers_clean.dtypes
})

print("Data Types Transformation:")
display(dtypes_comparison)

Pipeline Summary: Bronze vs Silver Layer
Original shape (Bronze): (99441, 5)
Cleaned shape (Silver): (99441, 5)
Total rows dropped: 0
Data Types Transformation:


,Bronze (Raw),Silver (Clean)
customer_id,object,string[python]
customer_unique_id,object,string[python]
customer_zip_code_prefix,int64,string[python]
customer_city,object,string[python]
customer_state,object,category


## 7. Data Quality Assertions & Export

Before persisting the clean dataset to the Silver layer, we execute automated data quality checks tailored to customer identity and geographic integrity:

* **Primary Key & Identity Integrity:** Ensure `customer_id` is unique and non-null, and `customer_unique_id` contains no missing values.
* **Mandatory Geographic Attributes:** Verify `customer_city`, `customer_state`, and `customer_zip_code_prefix` are fully populated with valid strings.
* **Geographic Domain Validity:** Validate that `customer_state` values strictly conform to the 27 official Brazilian state codes and `customer_zip_code_prefix` consists of 5-digit padded strings within the valid numeric range (01000–99999).
* **Sanitization Check:** Confirm that no placeholder values (`n/a`, `unknown`, empty strings) remain in location fields.
* **Dataset Validity:** Ensure the resulting dataset is non-empty before writing to storage.

In [8]:
assert customers_clean['customer_id'].is_unique, "Assertion failed: Duplicate customer_id found"
assert customers_clean['customer_id'].notna().all(), "Assertion failed: NULL values found in customer_id"
assert customers_clean['customer_unique_id'].notna().all(), "Assertion failed: NULL values found in customer_unique_id"

critical_cols = ['customer_zip_code_prefix', 'customer_city', 'customer_state']
for col in critical_cols:
    assert customers_clean[col].notna().all(), f"Assertion failed: NULL values found in mandatory column '{col}'"

valid_brazil_states = [
    'AC', 'AL', 'AP', 'AM', 'BA', 'CE', 'DF', 'ES', 'GO', 'MA',
    'MT', 'MS', 'MG', 'PA', 'PB', 'PR', 'PE', 'PI', 'RJ', 'RN',
    'RS', 'RO', 'RR', 'SC', 'SP', 'SE', 'TO'
]
invalid_states = ~customers_clean['customer_state'].isin(valid_brazil_states)
assert not invalid_states.any(), "Assertion failed: Unrecognized Brazilian state code found"

placeholders = ["", "-", "n/a", "null", "undefined", "unknown", "none"]
invalid_cities = customers_clean['customer_city'].str.lower().isin(placeholders)
assert not invalid_cities.any(), "Assertion failed: Placeholder or invalid values found in customer_city"

zip_ints = customers_clean['customer_zip_code_prefix'].astype(int)
invalid_zips = (zip_ints < 1000) | (zip_ints > 99999)
assert not invalid_zips.any(), "Assertion failed: ZIP code prefix outside valid 5-digit range"

assert len(customers_clean) > 0, "Assertion failed: Cleaned dataset is empty"

print("All quality checks passed successfully. Saving to Silver layer.")
customers_clean.to_parquet("../data/customers_clean.parquet", index=False)
print("Customers dataset successfully saved to Parquet.")

All quality checks passed successfully. Saving to Silver layer.
Customers dataset successfully saved to Parquet.
